# 🧩 Unsupervised Learning: High-Performance Clustering

Welcome to the FastKMeans Suite! In this notebook, we demonstrate the `FastKMeansClusterer`.
Unlike traditional K-Means, our clusterer supports **GPU acceleration**, **Stream Learning (EMA)**, and **Differentiable Regularization**.

We will use the real-world **Optical Recognition of Handwritten Digits** dataset.

In [ ]:
import torch
from sklearn.datasets import load_digits
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import homogeneity_score, completeness_score

from fast_kmeans import FastKMeansClusterer

def evaluate_clusters(X: torch.Tensor, y_true: torch.Tensor, k_clusters: int = 10) -> None:
    print(f"Initializing FastKMeansClusterer with {k_clusters} clusters...")
    
    clusterer = FastKMeansClusterer(
        k_init=k_clusters, 
        distance='cosine', 
        init_mode='kmeans++',
        # DIVERGENT REGULARIZATION: 
        # Orthogonality penalty that pushes centroids away from each other 
        # during Gradient Descent, completely preventing 'mode collapse'.
        diversity_reg=0.1 
    )
    
    print("--- Phase 1: Stream EMA Topology ---")
    clusterer.fit(X, y=None, max_iters=30, verbose=True)
    
    print("\n--- Phase 2: Gradient Optimization ---")
    clusterer.find_learning_rate(X)
    clusterer.finetune(X, y=None, epochs=20, verbose=True)
    
    preds = clusterer.predict(X).cpu().numpy()
    
    h_score = homogeneity_score(y_true.numpy(), preds)
    c_score = completeness_score(y_true.numpy(), preds)
    print(f"\n✅ Clustering Results:")
    print(f"Homogeneity Score: {h_score:.4f} (Perfect = 1.0)")
    print(f"Completeness Score: {c_score:.4f} (Perfect = 1.0)")

X_np, y_np = load_digits(return_X_y=True)
X_scaled = StandardScaler().fit_transform(X_np)
X_tensor = torch.tensor(X_scaled, dtype=torch.float32)
y_tensor = torch.tensor(y_np, dtype=torch.long)

evaluate_clusters(X_tensor, y_tensor, k_clusters=10)